In [1]:
import os
from pathlib import Path

for root in sorted(Path("/kaggle/input").glob("*")):
    print("===", root.name)
    count = 0
    with os.scandir(root) as entries:
        for entry in sorted(entries, key=lambda e: e.name):
            kind = "dir " if entry.is_dir() else "file"
            print(f"    {kind} {entry.name}")
            count += 1
            if count >= 15:
                print("    ... more not shown")
                break

=== competitions
    dir  rsna-knee-abnormality-detection
=== datasets
    dir  mohammedtalafha


In [2]:
import sys
from pathlib import Path

BASE = Path("/kaggle/input")
MINE = BASE / "datasets" if (BASE / "datasets").is_dir() else BASE
COMP = BASE / "competitions" / "rsna-knee-abnormality-detection"

# your code
CODE_ROOT = None
for marker in MINE.rglob("architecture.py"):
    if marker.parent.name == "model":
        CODE_ROOT = marker.parent.parent
        break
if CODE_ROOT is None:
    raise FileNotFoundError("could not find model/architecture.py in your datasets")

# your model
WANT = "model_pv2.pt"

found = {p.name: p for p in MINE.rglob("*.pt")}
if WANT not in found:
    raise FileNotFoundError(f"{WANT} is not attached. Found: {sorted(found)}")

MODEL_PATHS = [found[WANT]]
MODEL_PATH = MODEL_PATHS[0]

skipped = sorted(set(found) - {WANT})
if skipped:
    print("not submitting:", ", ".join(skipped))

# the competition data (top level only -- fast)
DATA_ROOT = COMP if (COMP / "test.csv").is_file() else None
if DATA_ROOT is None:
    for child in sorted(COMP.glob("*")):
        if child.is_dir() and (child / "test.csv").is_file():
            DATA_ROOT = child
            break
if DATA_ROOT is None:
    print("test.csv not found. Competition folder holds:")
    for item in sorted(COMP.glob("*"))[:20]:
        print("   ", "dir " if item.is_dir() else "file", item.name)
    raise FileNotFoundError("could not find test.csv")

sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT / "developments" / "src"))
print("code :", CODE_ROOT)
for path in MODEL_PATHS:
    print("model:", path)
print("data :", DATA_ROOT)

not submitting: model_finetuned.pt, model_frozen.pt
code : /kaggle/input/datasets/mohammedtalafha/cnn-cpc-code
model: /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_pv2.pt
data : /kaggle/input/competitions/rsna-knee-abnormality-detection


In [3]:
from model._implementation import read_config
from model.architecture import load

config = read_config(str(CODE_ROOT / "config" / "current_model.yaml"))
config["data_root"] = str(DATA_ROOT)

model, payload = load(str(MODEL_PATH), device="cpu")
print("encoder     :", payload.get("encoder_source", "report-aligned"))
print("epochs done :", payload.get("completed_epochs"))

if str(payload.get("encoder_source", "report-aligned")) == "dinov3":
    raise RuntimeError("this is the DINOv3 model -- it needs internet. Use the other one.")

del model

encoder     : report-aligned
epochs done : 2


In [4]:
from testing.test import predict_test_set

out = predict_test_set(
    config,
    checkpoint=str(MODEL_PATH),
    out_path="/kaggle/working/submission.csv",
)
print("wrote", out)

device=Tesla T4 | single-gpu | precision=fp16 | workers=2 | visible_gpus=2
[ensemble] loaded model_pv2.pt from frozen-all-lang
{
  "checkpoints": [
    "/kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_pv2.pt"
  ],
  "ensemble_size": 1,
  "checkpoint": "/kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_pv2.pt",
  "checkpoint_sha256": "0a76b9cab49a8f0a34a57ff68b707ed0cfb483b568360c5df5b8c96d006dc3f0",
  "checkpoint_sha256_all": [
    "0a76b9cab49a8f0a34a57ff68b707ed0cfb483b568360c5df5b8c96d006dc3f0"
  ],
  "submission_sha256": "1090921b554f0e12939a73b3d9940c4184336b052df4a0cd2b6f94b5df62ba1b",
  "completed_epochs": 2,
  "fixed_endpoint": true,
  "encoder_frozen": true,
  "encoder_sha256": "b328667cf9dfa9b909ef181c1bcc8975ec42bcd8b9eddad08f908875b73fae96",
  "expert_labels_in_gradients": 0,
  "crop_policy": {
    "version": "joint_focus_center_crop_only_v1",
    "crop_fraction": 0.9
  },
  "slice_offsets": [
    -1,
    0,
    1
  ],
  "test_studies": 3,
  "test_se

In [5]:
import pandas as pd

frame = pd.read_csv("/kaggle/working/submission.csv")
print("rows   :", len(frame))
print("columns:", len(frame.columns))
print(frame.head())

scores = frame.drop(columns=["StudyInstanceUID"])
spread = (scores.max() - scores.min()).sort_values()
print("\nsmallest spreads:")
print(spread.head(3))
if spread.max() < 0.01:
    print("\nWARNING: every column is nearly the same -- do not submit this")

rows   : 3
columns: 13
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.551617  0.086095   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.603640  0.094726   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.532827  0.083053   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.736012          0.122655   0.778541    0.655989  0.637772   
1         0.773202          0.143312   0.809925    0.707167  0.690924   
2         0.722998          0.113516   0.758550    0.636667  0.624695   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.621166   0.882347  0.566774   0.618534  0.477977  
1  0.673918   0.884431  0.620383   0.663301  0.542917  
2  0.591630   0.878898  0.554913   0.593038  0.453501  

smallest spreads:
Synovitis           0.005532
MCL                 0.011673
Lateral Meniscus    0.029796
dtype: float64


In [6]:
from pathlib import Path

BASE = Path("/kaggle/input")
MINE = BASE / "datasets" if (BASE / "datasets").is_dir() else BASE

print("files Kaggle can see:")
for p in sorted(MINE.rglob("*.pt")):
    print("   ", p)

print("\nMODEL_PATH is currently:", MODEL_PATH)

files Kaggle can see:
    /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_finetuned.pt
    /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_frozen.pt
    /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_pv2.pt

MODEL_PATH is currently: /kaggle/input/datasets/mohammedtalafha/frozen-all-lang/model_pv2.pt
